[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [PySpark & Parquet](PySpark-Parquet.md) | [Tasks](Tasks.md) | [Task 1](Task-1-aggregate-daily-traffic.md) | [Task 2](Task-2-daily-churn.md) | [Task 3](Task-3-batch-job.md) | [Task 4](Task-4-full-period-analysis.md) | [Prototype Notebook](ir_traffic_prototype.ipynb) | Analysis Notebook ⮕

# Iran (IR) traffic change analysis

This notebook builds on `ir_traffic_prototype.ipynb` and `ir_traffic_batch.py`. It does
**not** talk to Spark or S3 - everything here is read from the Parquet files that the
prototype notebook (test window, Tasks 1-2) and the Slurm batch job (full period, Task 3)
already wrote out.

Two parts:

1. **Test window (2026-02-14 to 2026-02-15)** - using the Parquet files under your Lustre
   project output directory (`/expanse/lustre/projects/<project>/<username>/output/prototype/`):
   top ASNs by packet count, top ASNs by number of distinct `/24`s, and the hourly traffic
   trend.
2. **Full period (2026-02-10 to 2026-03-05)** - using the combined daily summaries produced
   by the batch job: traffic over the whole period, and which days show a traffic drop.

Replace every `# YOUR CODE HERE` section below with your own code, then answer the
questions in the markdown cells that follow each task.

In [ ]:
import os
import glob

import pandas as pd
import matplotlib.pyplot as plt

## Environment variables

In [ ]:
#User name on Expanse (automatically set by the environment variable $USER)
USER=%env USER

%load_ext dotenv
%dotenv /expanse/lustre/projects/sdp178/kmok/.ucsdnts3.env

#Project ID on Expanse (preloaded in your shell - see Datasets.md)
PROJECT = %env ETP26_ACCOUNT

#Slurm Job ID of your Task 3 batch job array (for Part B) - see Tasks.md, Task 3 step 4
JOBID='YOUR_ARRAY_JOB_ID'

## Part A: test window (2026-02-14 to 2026-02-15)

Load the Parquet files the prototype notebook wrote to your Lustre project output
directory (`/expanse/lustre/projects/<project>/<username>/output/prototype/`):

- `ir_asn_daily_packets_test.parquet` - packets per ASN per day
- `ir_src_asn_test.parquet` - distinct `(src_ip, prefix2asn)` pairs per day
- `ir_hourly_packets_test.parquet` - packets and unique ASNs per hour

See the [Task 4 Guide](Task-4-full-period-analysis.md) for the pandas patterns
behind all three TODO cells in this notebook.

In [ ]:
PROTOTYPE_OUTPUT_DIR = f'/expanse/lustre/projects/{PROJECT}/{USER}/output/prototype'

asn_daily_packet_df = pd.read_parquet(
    os.path.join(PROTOTYPE_OUTPUT_DIR, 'ir_asn_daily_packets_test.parquet')
)
ir_src_asn_df = pd.read_parquet(
    os.path.join(PROTOTYPE_OUTPUT_DIR, 'ir_src_asn_test.parquet')
)
hourly_packet_df = pd.read_parquet(
    os.path.join(PROTOTYPE_OUTPUT_DIR, 'ir_hourly_packets_test.parquet')
).sort_values('hour').reset_index(drop=True)

print(f'{len(asn_daily_packet_df)} ASN/day rows, {len(ir_src_asn_df)} src_ip/ASN rows, '
      f'{len(hourly_packet_df)} hourly rows')

### 1. Top 5 ASNs by packet count (both test days combined)

Group `asn_daily_packet_df` by `prefix2asn`, sum `total_packet_cnt` across both
test days, and take the top 5.

In [ ]:
# YOUR CODE HERE
# top5_asn_by_packets = ...
top5_asn_by_packets

### 2. Top 5 ASNs by number of distinct `/24` prefixes

Same idea as the day-over-day churn cell in the prototype notebook: bucket each
distinct source IP down to its `/24`, dedup `(subnet_24, prefix2asn)` pairs across
both test days, then count how many distinct `/24`s each ASN contributes.

In [ ]:
def ip_to_slash24(ip):
    return '.'.join(ip.split('.')[:3]) + '.0/24'

# YOUR CODE HERE
# top5_asn_by_subnet24 = ...
top5_asn_by_subnet24

**Q4:** Are the top 5 ASNs by packet count the same as the top 5 by distinct `/24`
count? What would it mean for an ASN to rank high on one metric but not the other?

_Your answer here._

### 3. Hourly traffic trend

Total packets and unique ASNs per hour, from `hourly_packet_df`. Plotted as two
stacked panels since the two metrics are on very different scales.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 7), sharex=True)

ax1.plot(
    hourly_packet_df['hour'], hourly_packet_df['total_packet_cnt'],
    color='#2a78d6', linewidth=2, marker='o', markersize=5,
)
ax1.set_title('Hourly inbound telescope traffic attributed to Iran (IR) - test window')
ax1.set_ylabel('Total packets from IR sources')
ax1.grid(axis='y', color='#e1e0d9', linewidth=1)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2.plot(
    hourly_packet_df['hour'], hourly_packet_df['unique_asns'],
    color='#2a78d6', linewidth=2, marker='o', markersize=5,
)
ax2.set_title('Hourly unique ASNs attributed to Iran (IR) - test window')
ax2.set_xlabel('Hour (UTC)')
ax2.set_ylabel('Unique ASNs')
ax2.grid(axis='y', color='#e1e0d9', linewidth=1)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

**Q5:** Is there a diurnal (day/night) pattern in the hourly trend? Is it driven
by a few ASNs sending more traffic at certain hours, or by more/fewer distinct
ASNs being active - and does it line up with Iran's local time (UTC+3:30)?

_Your answer here._

## Part B: full period (2026-02-10 to 2026-03-05), from the Slurm batch job

Set `JOBID` in the environment-variables cell near the top of this notebook to the
array job id from Task 3 - `BATCH_OUTPUT_DIR` below then resolves to the directory
`run_ir_traffic.slurm` wrote to (printed by the job / visible via `squeue`):

```python
BATCH_OUTPUT_DIR = f"/expanse/lustre/projects/{PROJECT}/{USER}/output/ir_traffic/{JOBID}"
```

Then combine the 24 per-day `ir_daily_summary_<date>.parquet`, `ir_src_asn_<date>.parquet`,
and `ir_asn_daily_packets_<date>.parquet` files into one DataFrame each, covering the
whole period, and write those combined DataFrames back out to Parquet.

In [ ]:
# Set JOBID in the environment-variables cell near the top of this notebook to your
# Task 3 array job id, then this resolves to the directory your job array wrote to.
BATCH_OUTPUT_DIR = f"/expanse/lustre/projects/{PROJECT}/{USER}/output/ir_traffic/{JOBID}"

summary_files = sorted(glob.glob(os.path.join(BATCH_OUTPUT_DIR, 'ir_daily_summary_*.parquet')))
src_asn_files = sorted(glob.glob(os.path.join(BATCH_OUTPUT_DIR, 'ir_src_asn_*.parquet')))
asn_daily_files = sorted(glob.glob(os.path.join(BATCH_OUTPUT_DIR, 'ir_asn_daily_packets_*.parquet')))

full_daily_summary_df = pd.concat((pd.read_parquet(f) for f in summary_files), ignore_index=True)
full_daily_summary_df['date'] = pd.to_datetime(full_daily_summary_df['date'])
full_daily_summary_df = full_daily_summary_df.sort_values('date').reset_index(drop=True)

full_src_asn_df = pd.concat((pd.read_parquet(f) for f in src_asn_files), ignore_index=True)
# The distinct list of IR-geolocated source IPs/ASNs across the whole period,
# independent of which day(s) each one was observed on.
full_src_asn_unique_df = full_src_asn_df.drop_duplicates(subset=['src_ip', 'prefix2asn']).reset_index(drop=True)

full_asn_daily_packet_df = pd.concat((pd.read_parquet(f) for f in asn_daily_files), ignore_index=True)
full_asn_daily_packet_df['date'] = pd.to_datetime(full_asn_daily_packet_df['date'])
full_asn_daily_packet_df = full_asn_daily_packet_df.sort_values(
    ['date', 'total_packet_cnt'], ascending=[True, False]
).reset_index(drop=True)

full_daily_summary_df.to_parquet('ir_daily_summary_full.parquet', index=False)
full_src_asn_unique_df.to_parquet('ir_src_asn_full.parquet', index=False)
full_asn_daily_packet_df.to_parquet('ir_asn_daily_packets_full.parquet', index=False)

print(f'{len(full_daily_summary_df)} days combined -> ir_daily_summary_full.parquet')
print(f'{len(full_src_asn_unique_df)} unique IR source IP/ASN pairs -> ir_src_asn_full.parquet')
print(f'{len(full_asn_daily_packet_df)} day/ASN rows combined -> ir_asn_daily_packets_full.parquet')
full_daily_summary_df

### 1. Daily IR-attributed traffic across the full period

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.plot(
    full_daily_summary_df['date'], full_daily_summary_df['total_packet_cnt'],
    color='#2a78d6', linewidth=2, marker='o', markersize=4,
)
ax.set_title('Daily inbound telescope traffic attributed to Iran (IR): 2026-02-10 to 2026-03-05')
ax.set_xlabel('Date')
ax.set_ylabel('Total packets from IR sources')
ax.grid(axis='y', color='#e1e0d9', linewidth=1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

### 2. Which days show a traffic drop?

Flag a day as a "drop" if its total packet count falls by more than `DROP_THRESHOLD`
(50%) compared to the previous day. `pandas.Series.pct_change()` is useful here.

In [ ]:
DROP_THRESHOLD = 0.5  # flag a day if its traffic falls by more than 50% vs. the previous day

# YOUR CODE HERE
# drop_days_df = ...
print(f'{len(drop_days_df)} day(s) flagged as a traffic drop (> {DROP_THRESHOLD:.0%} drop vs. previous day)')
drop_days_df[['date', 'total_packet_cnt', 'pct_change_from_prev_day']]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.plot(
    full_daily_summary_df['date'], full_daily_summary_df['total_packet_cnt'],
    color='#2a78d6', linewidth=2, marker='o', markersize=4, label='Daily total packets',
)
ax.scatter(
    drop_days_df['date'], drop_days_df['total_packet_cnt'],
    color='#d64545', s=80, zorder=5, label='Flagged drop day',
)
ax.set_title('Daily IR-attributed traffic with flagged drop days')
ax.set_xlabel('Date')
ax.set_ylabel('Total packets from IR sources')
ax.grid(axis='y', color='#e1e0d9', linewidth=1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend()

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

In [ ]:
# Check whether the flagged drop day(s) hit most ASNs or a concentrated few
# (full_asn_daily_packet_df was already combined above).
full_asn_daily_packet_df[full_asn_daily_packet_df['date'].isin(drop_days_df['date'])].head(10)

### 3. Background: correlate the drops with the 2026 Iran internet blackout

Read the background on the **[2026 Internet blackout in Iran](https://en.wikipedia.org/wiki/2026_Internet_blackout_in_Iran)**
Wikipedia article before answering.

**Q6:** Do the day(s) you flagged as a traffic drop (`drop_days_df`) fall within the
blackout dates reported by the article? Does the size and duration of your measured
drop match how the article describes the blackout's severity and scope (e.g.
nationwide vs. regional, full shutdown vs. throttling)? Use the table above
(`full_asn_daily_packet_df` on the flagged day(s)) to check whether the drop hit
most ASNs or a concentrated few, and factor that into your answer.

_Your answer here._

[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [PySpark & Parquet](PySpark-Parquet.md) | [Tasks](Tasks.md) | [Task 1](Task-1-aggregate-daily-traffic.md) | [Task 2](Task-2-daily-churn.md) | [Task 3](Task-3-batch-job.md) | [Task 4](Task-4-full-period-analysis.md) | [Prototype Notebook](ir_traffic_prototype.ipynb) | Analysis Notebook ⮕